#Building a english - french language translation model using encoder decoder with lstm networks

- we are building seq2seq neural network here


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input,Embedding,LSTM,Dense



In [ ]:
#step-2

eng  = ["hi","hello","how are you","thank you"]
fr = ['salut','bonjour','comment ça va','merci']

In [ ]:
#step-3 - for english

tok_eng = Tokenizer()
tok_eng.fit_on_texts(eng) #learn all eng unique words
x = tok_eng.texts_to_sequences(eng) #convert each sentenc into interger word id
x = pad_sequences(x)


In [ ]:
x

array([[0, 0, 2],
       [0, 0, 3],
       [4, 5, 1],
       [0, 6, 1]], dtype=int32)

In [ ]:
#step-4 - for french

tok_fr = Tokenizer()
tok_fr.fit_on_texts(fr) #learn all eng unique words
y = tok_fr.texts_to_sequences(fr) #convert each sentenc into interger word id
y = pad_sequences(y)

In [ ]:
y

array([[0, 0, 1],
       [0, 0, 2],
       [3, 4, 5],
       [0, 0, 6]], dtype=int32)

In [ ]:
y_in = y[:,:-1] #removesthe last word -- input to decoder
y_out = y[:,1:]# removes the first word -- target output

In [ ]:
y_in

array([[0, 0],
       [0, 0],
       [3, 4],
       [0, 0]], dtype=int32)

In [ ]:
y_out

array([[0, 1],
       [0, 2],
       [4, 5],
       [0, 6]], dtype=int32)

neural networks expect 3d data for seq tasks

In [ ]:
y_out = y_out.reshape((y_out.shape[0],y_out.shape[1],1))

In [ ]:
y_out

array([[[0],
        [1]],

       [[0],
        [2]],

       [[4],
        [5]],

       [[0],
        [6]]], dtype=int32)

In [ ]:
vocab_eng = len(tok_eng.word_index)+1
vocab_fr =len(tok_fr.word_index)+1

In [ ]:
#build the encoder

enc_in = Input(shape = (x.shape[1],))
enc_emb = Embedding(vocab_eng,8)(enc_in)
_,h,c = LSTM(32, return_state= True)(enc_emb)

In [ ]:
#build a decoder
dec_in = Input(shape = (y_in.shape[1],))
dec_emb = Embedding(vocab_fr,8)(dec_in)
dec_out,_,_ = LSTM(32, return_sequences= True, return_state= True)(dec_emb , initial_state = [h,c])
out = Dense(vocab_fr,activation = 'softmax')(dec_out)


In [ ]:
model = Model([enc_in,dec_in],out)
model.compile(optimizer ='adam',loss = 'sparse_categorical_crossentropy')

In [ ]:
model.fit([x,y_in],y_out,epochs = 300,verbose = 0)

In [ ]:
preds = model.predict([x,y_in])

for i , pred in enumerate(preds):
  ids = np.argmax(pred,axis =1)
  words = [tok_fr.index_word.get(idx,'???') for idx in ids ]
  print(f"english : {eng[i]} ------------->>>> predicted french: {''.join(words)}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 192ms/step
english : hi ------------->>>> predicted french: ???salut
english : hello ------------->>>> predicted french: ???bonjour
english : how are you ------------->>>> predicted french: çava
english : thank you ------------->>>> predicted french: ???merci
